In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture

In [2]:
def matriz_conf(y_true, y_pred, labels):
    total_labels = labels
    print(total_labels)
    cm = np.zeros((len(total_labels),len(total_labels)), dtype=int)
    for i in range(len(y_true)):
        cm[y_true[i]][y_pred[i]] += 1

    
    cm = pd.DataFrame(cm, columns=total_labels, index=total_labels)

    cm_transp = pd.DataFrame(np.transpose(cm.to_numpy()), columns=total_labels, index=total_labels)

    for c in cm_transp.columns:
        cm_transp[c] = cm_transp[c]/cm_transp[c].sum()

    cm_porcento = pd.DataFrame(np.transpose(cm_transp.to_numpy()), columns=total_labels, index=total_labels)

    return cm, cm_porcento

def acc(cm, hidden_classes):
    cm_transp = pd.DataFrame(np.transpose(cm.dropna().to_numpy()), columns=cm.columns, index=cm.columns)
    acc = 0
    total = 0
    for c in cm_transp.columns:
        if c not in hidden_classes:
            acc += cm_transp[c][c]
        else:
            acc += cm_transp[c][-1]
        total += cm_transp[c].sum()
    return acc/total

In [3]:
filenames = [0,2,3,4,5]

labels_str = ['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']

filenames

# pd.set_option('future.no_silent_downcasting', True)

[0, 2, 3, 4, 5]

In [4]:
exp_test = []
y_true_all_exp_test = []
for i in range(len(filenames)):
    exp_test.append(pd.read_csv(f'test_{filenames[i]}_GMM_BIC_1_10_scores_corr.csv'))
    y_true_all_exp_test.append(exp_test[i]['Label'].values.tolist())
    exp_test[i] = exp_test[i].drop(columns=['Label'])

In [6]:
from sklearn.metrics import classification_report
ths = [21]
f1s = [[]]
for i in range(len(exp_test)):
    index = 0
    for th in ths:
        y_pred = []
        prog= 0
        for j in range(len(exp_test[i])):
            # print(exp[i][j])
            m = np.nanmax(exp_test[i].values[j])
            # print(m)
            if m > th:
                y_pred.append(exp_test[i].values[j].tolist().index(m))
            else:
                y_pred.append(-1)
        # print(y_pred[:10])

        cm, cm_porcento = matriz_conf(y_true_all_exp_test[i], y_pred, list(range(len(labels_str))) + [-1])
        print(f'th = {th} hidden = {filenames[i]}')
        display(cm)
        tp = cm[-1][filenames[i]]
        fp = cm[-1].sum() - tp
        fn = cm.iloc[filenames[i]].sum() - tp
        tn = cm.drop(columns=[-1]).values.sum() - fn

        acc = (tp+tn)/(tp+fp+tn+fn)
        recall = tp/(tp+fn)
        precision = tp/(tp+fp)
        if precision == 0 or recall == 0:
            f1 = 0
        else:
            f1 = 2*precision*recall/(precision+recall)

        f1s[index].append(f1)
        index += 1
        print(f'th: {th} hidden: {filenames[i]} acc:{acc} recall:{recall} precision:{precision} f1:{f1}')

        true_labels = np.array(y_true_all_exp_test[i])
        true_labels[true_labels == filenames[i]] = -1

        print('MULTICLASS DETECTION')
        print(classification_report(true_labels, y_pred))

[0, 1, 2, 3, 4, 5, -1]
th = 21 hidden = 0


,0,1,2,3,4,5,-1
0,0,66099,0,0,0,0,186687
1,0,155566,28,7,355,76,4638
2,0,0,102449,0,0,0,432
3,0,0,1,76129,0,0,57
4,0,74,0,0,56970,2,192
5,0,10,0,0,12,105,58
-1,0,0,0,0,0,0,0


th: 21 hidden: 0 acc:0.8900279561256533 recall:0.7385179558994565 precision:0.9720041236254582 f1:0.8393256153759693
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.97      0.74      0.84    252786
           1       0.70      0.97      0.81    160670
           2       1.00      1.00      1.00    102881
           3       1.00      1.00      1.00     76187
           4       0.99      1.00      0.99     57238
           5       0.57      0.57      0.57       185

    accuracy                           0.89    649947
   macro avg       0.87      0.88      0.87    649947
weighted avg       0.91      0.89      0.89    649947

[0, 1, 2, 3, 4, 5, -1]
th = 21 hidden = 2


,0,1,2,3,4,5,-1
0,251905,356,0,0,0,0,525
1,39,141174,0,15,133,67,19242
2,0,48,0,1576,0,0,101257
3,0,0,0,76063,0,0,124
4,1,6,0,0,56587,0,644
5,2,6,0,0,8,101,68
-1,0,0,0,0,0,0,0


th: 21 hidden: 2 acc:0.9658018269181948 recall:0.9842147724069557 precision:0.8309289348432628 f1:0.9010994878549087
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.83      0.98      0.90    102881
           0       1.00      1.00      1.00    252786
           1       1.00      0.88      0.93    160670
           3       0.98      1.00      0.99     76187
           4       1.00      0.99      0.99     57238
           5       0.60      0.55      0.57       185

    accuracy                           0.96    649947
   macro avg       0.90      0.90      0.90    649947
weighted avg       0.97      0.96      0.97    649947

[0, 1, 2, 3, 4, 5, -1]
th = 21 hidden = 3


...

th: 21 hidden: 3 acc:0.9536331423946875 recall:0.9997243624240356 precision:0.7166473781767202 f1:0.8348422737137471
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.72      1.00      0.83     76187
           0       1.00      0.99      0.99    252786
           1       1.00      0.85      0.92    160670
           2       1.00      0.97      0.99    102881
           4       1.00      0.99      0.99     57238
           5       0.20      0.67      0.31       185

    accuracy                           0.95    649947
   macro avg       0.82      0.91      0.84    649947
weighted avg       0.97      0.95      0.95    649947

[0, 1, 2, 3, 4, 5, -1]
th = 21 hidden = 4


,0,1,2,3,4,5,-1
0,251734,2,0,0,0,0,1050
1,2345,144720,10,0,0,1518,12077
2,0,0,101077,0,0,0,1804
3,0,0,1,76114,0,0,72
4,1,3003,0,0,0,4,54230
5,1,0,0,0,0,118,66
-1,0,0,0,0,0,0,0


th: 21 hidden: 4 acc:0.9721869629369779 recall:0.9474474999126454 precision:0.7825509747615406 f1:0.8571405991923311
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.78      0.95      0.86     57238
           0       0.99      1.00      0.99    252786
           1       0.98      0.90      0.94    160670
           2       1.00      0.98      0.99    102881
           3       1.00      1.00      1.00     76187
           5       0.07      0.64      0.13       185

    accuracy                           0.97    649947
   macro avg       0.80      0.91      0.82    649947
weighted avg       0.97      0.97      0.97    649947

[0, 1, 2, 3, 4, 5, -1]
th = 21 hidden = 5


,0,1,2,3,4,5,-1
0,251321,0,0,0,0,0,1465
1,865,141892,11,8,211,0,17683
2,0,0,100570,0,0,0,2311
3,0,0,1,76112,0,0,74
4,1,51,0,0,56793,0,393
5,2,76,0,0,10,0,97
-1,0,0,0,0,0,0,0


th: 21 hidden: 5 acc:0.9661295459475927 recall:0.5243243243243243 precision:0.004404486218952913 f1:0.008735590778097982
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.52      0.01       185
           0       1.00      0.99      1.00    252786
           1       1.00      0.88      0.94    160670
           2       1.00      0.98      0.99    102881
           3       1.00      1.00      1.00     76187
           4       1.00      0.99      0.99     57238

    accuracy                           0.96    649947
   macro avg       0.83      0.90      0.82    649947
weighted avg       1.00      0.96      0.98    649947



# Média absolute threshold

In [7]:
for i in range(len(f1s)):
    print(f'Média f1 absolute th {ths[i]}: {np.mean(np.array(f1s[i]))}')

Média f1 absolute th 21: 0.6882287133830108
